## 6. Save for Ensemble

✅ Random Forest predictions ready for ensemble combination

**Output Summary:**
- Predictions shape: (96,) - matches test set
- Scale: [0, 1] range (scaled)
- Format: Numpy array
- Metric (R²): {:.4f}

These predictions will be combined with LSTM, RNN, and Gaussian Process predictions in the ensemble notebook.

**Next:** Run the Gaussian notebook to generate its predictions.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Actual vs Predicted (first 100 samples)
axes[0].plot(y_test_flat[:100], label='Actual', linewidth=2, alpha=0.7, color='#2E86AB')
axes[0].plot(rf_predictions_scaled[:100], label='RF Predicted', linewidth=2, alpha=0.7, color='#F18F01')
axes[0].set_title('Random Forest: Actual vs Predicted (First 100 samples)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Scaled Demand')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = y_test_flat - rf_predictions_scaled
axes[1].plot(residuals, label='Residuals', linewidth=1, alpha=0.7, color='#A23B72')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].fill_between(range(len(residuals)), residuals, alpha=0.3, color='#A23B72')
axes[1].set_title('Random Forest: Prediction Residuals', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Prediction visualizations created")

## 5. Visualize Predictions

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': [f'feature_{i}' for i in range(model.n_features_in_)],
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(feature_importance['feature'], feature_importance['importance'], color='#2E86AB')
ax.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax.set_title('Random Forest - Feature Importance', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 5 Important Features:")
print(feature_importance.head())

## 4. Feature Importance

In [ ]:
# Generate predictions (already in [0, 1] range from training data)
rf_predictions_scaled = model.predict(X_test_flat)

# Inverse scale to original range
rf_predictions = preprocessing.inverse_scale_predictions(rf_predictions_scaled, scaler_y)

print(f"Random Forest Predictions shape: {rf_predictions.shape}")
print(f"Random Forest Predictions range: [{rf_predictions.min():.2f}, {rf_predictions.max():.2f}]")
print(f"\nFirst 10 predictions: {rf_predictions[:10]}")

# Calculate metrics
rf_mse = mean_squared_error(y_test_flat, rf_predictions_scaled)
rf_mae = mean_absolute_error(y_test_flat, rf_predictions_scaled)
rf_r2 = r2_score(y_test_flat, rf_predictions_scaled)

print(f"\n📈 Random Forest Model Performance (on scaled data):")
print(f"  MSE: {rf_mse:.4f}")
print(f"  MAE: {rf_mae:.4f}")
print(f"  R²:  {rf_r2:.4f}")

## 3. Generate Predictions

In [ ]:
print("🚀 Training Random Forest model...")

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

model.fit(X_train_flat, y_train_flat)

print("✓ Training completed")
print(f"✓ Model features: {model.n_features_in_}")
print(f"✓ Trees trained: {model.n_estimators}")

## 2. Train Random Forest Model

In [ ]:
# Load data (SAME as all other models)
X, y = preprocessing.load_synthetic_data(n_samples=500, n_features=10)

# Unified preprocessing
data = preprocessing.preprocess_data(X, y, test_size=0.2, seq_length=10)

# Extract FLATTENED data for Random Forest
X_train_flat = data['X_train_flat']
X_test_flat = data['X_test_flat']
y_train_flat = data['y_train_flat']
y_test_flat = data['y_test_flat']
scaler_y = data['scaler_y']

print(f"Training data shapes (for Random Forest):")
print(f"  X_train: {X_train_flat.shape} (samples, features) - FLATTENED")
print(f"  y_train: {y_train_flat.shape}")
print(f"\nTest data shapes (for Random Forest):")
print(f"  X_test: {X_test_flat.shape}")
print(f"  y_test: {y_test_flat.shape}")

# IMPORTANT: Verify output dimension
print(f"\n✓ Test predictions will have shape: ({len(y_test_flat)},)")

## 1. Load and Preprocess Data

In [ ]:
import sys
sys.path.insert(0, '../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src import preprocessing
from src.models import random_forest_model

# Set random seed
np.random.seed(42)

print("✓ Libraries and modules imported successfully")

# Demand Forecasting System - Random Forest Model

## Overview
This notebook trains a **Random Forest Regressor** for demand forecasting using scikit-learn.

**Key Features:**
- Input shape: (n_samples, n_features) - flattened
- Output shape: (n_test_samples,) - **MUST match other models**
- 100 decision trees with max depth of 15
- Output: predictions in [0, 1] range (scaled)

**Critical:** All predictions must have the **same shape and scale** for ensemble averaging.